Завдання 1: Для кожної з адміністративних одиниць України завантажити (urllib) тестові структуровані файли, що містять значення VHI-індексу, додати дату та час завантаження і передбачити механізм запобігання повторного завантаження

In [1]:
import urllib.request
import datetime
import os

def download_noaa_data(province_id, start_year=1981, end_year=2026):
    os.makedirs('vhi_data', exist_ok=True)

    existing_files = [f for f in os.listdir('vhi_data') if f.startswith(f"vhi_id_{province_id}_")]
    if existing_files:
        print(f"Дані для області {province_id} вже існують: {existing_files[0]}")
        return f"vhi_data/{existing_files[0]}"

    url = f"https://www.star.nesdis.noaa.gov/smcd/emb/vci/VH/get_TS_admin.php?country=UKR&provinceID={province_id}&year1={start_year}&year2={end_year}&type=Mean"

    now = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    filename = f"vhi_data/vhi_id_{province_id}_{now}.csv"
    
    try:
        print(f"Завантаження даних для області {province_id}...")
        urllib.request.urlretrieve(url, filename)
        print(f"Успішно збережено: {filename}")
        return filename
    except Exception as e:
        print(f"Помилка завантаження для області {province_id}: {e}")
        return None

for i in range(1, 28):
    download_noaa_data(i)

Завантаження даних для області 1...
Успішно збережено: vhi_data/vhi_id_1_2026-06-15_02-40-45.csv
Завантаження даних для області 2...
Успішно збережено: vhi_data/vhi_id_2_2026-06-15_02-41-01.csv
Завантаження даних для області 3...
Успішно збережено: vhi_data/vhi_id_3_2026-06-15_02-41-02.csv
Завантаження даних для області 4...
Успішно збережено: vhi_data/vhi_id_4_2026-06-15_02-41-07.csv
Завантаження даних для області 5...
Успішно збережено: vhi_data/vhi_id_5_2026-06-15_02-41-07.csv
Завантаження даних для області 6...
Успішно збережено: vhi_data/vhi_id_6_2026-06-15_02-41-08.csv
Завантаження даних для області 7...
Успішно збережено: vhi_data/vhi_id_7_2026-06-15_02-41-12.csv
Завантаження даних для області 8...
Успішно збережено: vhi_data/vhi_id_8_2026-06-15_02-41-13.csv
Завантаження даних для області 9...
Успішно збережено: vhi_data/vhi_id_9_2026-06-15_02-41-18.csv
Завантаження даних для області 10...
Успішно збережено: vhi_data/vhi_id_10_2026-06-15_02-41-19.csv
Завантаження даних для облас

Завдання 1.2: Зчитати завантажені текстові файли у pandas dataframe. Здійснити data cleaning: прибрати зайві стовпці, заповнити пропуски, видалити зайвий текст тощо. Додати стовпчики з назвою та індексом області 
Реалізувати процедуру зміни індексів: в завантажених з NOAA даних області індексуються за англійською абеткою (Province 1 - Cherkasy), потрібно замінити індекси так, щоб області індексувалася за українською абеткою (1 область - Вінницька). 


In [24]:
import pandas as pd
import os

province_mapping = {
    1: 22, 2: 24, 3: 23, 4: 25, 5: 3, 6: 4, 7: 8, 8: 19, 9: 20, 10: 21,
    11: 9, 12: 26, 13: 10, 14: 11, 15: 12, 16: 13, 17: 14, 18: 15, 19: 16,
    20: 27, 21: 17, 22: 18, 23: 6, 24: 1, 25: 2, 26: 7, 27: 5
}

province_names = {
    1: 'Вінницька', 2: 'Волинська', 3: 'Дніпропетровська', 4: 'Донецька',
    5: 'Житомирська', 6: 'Закарпатська', 7: 'Запорізька', 8: 'Івано-Франківська',
    9: 'Київська', 10: 'Кіровоградська', 11: 'Луганська', 12: 'Львівська',
    13: 'Миколаївська', 14: 'Одеська', 15: 'Полтавська', 16: 'Рівненська',
    17: 'Сумська', 18: 'Тернопільська', 19: 'Харківська', 20: 'Херсонська',
    21: 'Хмельницька', 22: 'Черкаська', 23: 'Чернівецька', 24: 'Чернігівська',
    25: 'Республіка Крим', 26: 'м. Київ', 27: 'м. Севастополь'
}

all_data = []

for filename in os.listdir('vhi_data'):
    if filename.endswith('.csv'):
        filepath = os.path.join('vhi_data', filename)
        old_id = int(filename.split('_')[2])
        
        with open(filepath, 'r') as f:
            lines = f.readlines()
        
        parsed_rows = []
        for line in lines:
            clean_line = line.replace('<tt>', '').replace('</tt>', '').replace('<br>', '').replace('<pre>', '').replace('</pre>', '').strip()  
            parts = [p.strip() for p in clean_line.split(',')]
            if len(parts) >= 7 and parts[0].isdigit() and len(parts[0]) == 4:
                try:
                    year = int(parts[0])
                    week = int(parts[1])
                    smn = float(parts[2])
                    smt = float(parts[3])
                    vci = float(parts[4])
                    tci = float(parts[5])
                    vhi = float(parts[6])
                    if vhi != -1.0:
                        parsed_rows.append([year, week, smn, smt, vci, tci, vhi, old_id])
                except ValueError:
                    continue
        df = pd.DataFrame(parsed_rows, columns=['Year', 'Week', 'SMN', 'SMT', 'VCI', 'TCI', 'VHI', 'area'])
        all_data.append(df)
df_full = pd.concat(all_data, ignore_index=True)
df_full['Province_ID'] = df_full['area'].map(province_mapping)
df_full['Province_Name'] = df_full['Province_ID'].map(province_names)
df_full = df_full[['Year', 'Week', 'SMN', 'SMT', 'VCI', 'TCI', 'VHI', 'Province_ID', 'Province_Name']]

print(f"Таблиця успішно сформована! Загальна кількість записів: {len(df_full)}")
display(df_full.head())

Таблиця успішно сформована! Загальна кількість записів: 61047


,Year,Week,SMN,SMT,VCI,TCI,VHI,Province_ID,Province_Name
0,1982,1,0.059,258.24,51.11,48.78,49.95,21,Хмельницька
1,1982,2,0.063,261.53,55.89,38.20,47.04,21,Хмельницька
2,1982,3,0.063,263.45,57.30,32.69,44.99,21,Хмельницька
3,1982,4,0.061,265.10,53.96,28.62,41.29,21,Хмельницька
4,1982,5,0.058,266.42,46.87,28.57,37.72,21,Хмельницька


Завдання 1.3
Реалізувати процедури для формування вибірок:
Ряд VHI для області за вказаний рік;
Ряд VHI за вказаний діапазон років для вказаних областей;
Пошук екстремумів (min та max) для вказаних областей та років, середнього, медіани;
Інші вибірки на вимогу викладача.
Всі процедури необхідно реалізувати окремими функціями та викликати їх в окремих комірках Jupyter Notebook. Додати текст завдань перед кожним пунктом та читабельний вивід функцій. 


In [25]:
def get_vhi_series(df, province_id, year):
    """Повертає ряд VHI для області за вказаний рік."""
    return df[(df['Province_ID'] == province_id) & (df['Year'] == year)][['Week', 'VHI']]

def get_vhi_range(df, province_id, start_year, end_year):
    """Повертає ряд VHI за вказаний діапазон років для вказаної області."""
    return df[(df['Province_ID'] == province_id) & (df['Year'] >= start_year) & (df['Year'] <= end_year)][['Year', 'Week', 'VHI']]

def analyze_vhi(df, province_id, year):
    """Пошук екстремумів (min, max), середнього та медіани."""
    subset = df[(df['Province_ID'] == province_id) & (df['Year'] == year)]
    
    if subset.empty:
        return "Даних немає"
        
    return {
        "Мінімум": subset['VHI'].min(),
        "Максимум": subset['VHI'].max(),
        "Середнє": round(subset['VHI'].mean(), 2),
        "Медіана": subset['VHI'].median()
    }

In [26]:

test_province = 1
test_year = 2000

print(f"=== 1. Ряд VHI для області {test_province} за {test_year} рік (перші 5 тижнів) ===")
display(get_vhi_series(df_full, test_province, test_year).head())

print(f"\n=== 2. Ряд VHI для області {test_province} за діапазон 2000-2002 років (перші 5 записів) ===")
display(get_vhi_range(df_full, test_province, 2000, 2002).head())

print(f"\n=== 3. Статистика VHI для області {test_province} за {test_year} рік ===")
stats = analyze_vhi(df_full, test_province, test_year)
if isinstance(stats, dict):
    for key, value in stats.items():
        print(f"{key}: {value}")
else:
    print(stats)

=== 1. Ряд VHI для області 1 за 2000 рік (перші 5 тижнів) ===


,Week,VHI
34821,1,24.22
34822,2,27.70
34823,3,30.68
34824,4,32.55
34825,5,34.73



=== 2. Ряд VHI для області 1 за діапазон 2000-2002 років (перші 5 записів) ===


,Year,Week,VHI
34821,2000,1,24.22
34822,2000,2,27.70
34823,2000,3,30.68
34824,2000,4,32.55
34825,2000,5,34.73



=== 3. Статистика VHI для області 1 за 2000 рік ===
Мінімум: 11.25
Максимум: 63.27
Середнє: 40.98
Медіана: 40.91
